# EDA of Women's Dress Attributes and Sales

> **Project type: guided coursework exercise, not an independent project.** The datasets, the notebook skeleton and the question prompts were supplied by a course (**Upgrad:PG Diploma in ML and Data SCience**). This notebook is my worked solution to that exercise, plus a later cleanup pass.

| | |
|---|---|
| **Problem** | A retailer's dress catalogue comes as two messy tables: *attributes* (style, price band, size, season, neckline, material, rating, recommendation flag) and *sales* (one column per date). Clean and merge them, then use univariate, bivariate and multivariate EDA to describe what sells, what is rated well and what is recommended. |
| **Dataset / source** | UCI Machine Learning Repository, *Dresses_Attribute_Sales*: Usman, M. & Ahmed, A. (2014), DOI [10.24432/C56W3V](https://doi.org/10.24432/C56W3V), licence CC BY 4.0. This notebook uses the course's CSV copies (`Attribute+DataSet.csv`, `Dress+Sales.csv`, 479 dresses each), which may differ from the 500/501-row UCI release. |
| **Provided by the course** | The two CSV files, the section structure, the multiple-choice prompts (paraphrased below, not copied) and the `# Print ...` cell hints. |
| **My contribution** | Code and answers for each prompt across cleaning, univariate, bivariate and multivariate analysis. Later cleanup (AI-assisted with Claude): bug fixes, consolidation, the sensitivity checks, and the findings / limitations write-up. See the change log at the end. |
| **Approach** | (1) Profile types and missing values. (2) Standardise labels and convert sales columns to numbers. (3) Aggregate date columns into four seasons using the course's mapping. (4) Left-merge sales onto attributes. (5) Drop / label missing values. (6) Group rare categories into "Others". (7) Univariate, bivariate and multivariate EDA. |
| **Scope** | Descriptive EDA only. No modelling, no significance tests, no causal claims. |

## 0. Setup

In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")

DATA_DIR = Path(".")                      # change if the CSVs live elsewhere
SALES_CSV = DATA_DIR / "Dress+Sales.csv"
ATTR_CSV = DATA_DIR / "Attribute+DataSet.csv"
for f in (SALES_CSV, ATTR_CSV):
    assert f.exists(), f"Missing {f}. See the README ('Data') for where to get the files."

sales_raw = pd.read_csv(SALES_CSV)        # one row per dress, one column per date
attr_raw = pd.read_csv(ATTR_CSV)          # one row per dress, descriptive attributes
print("Dress Sales  :", sales_raw.shape)
print("Attributes   :", attr_raw.shape)
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

*Naming:* the course template called these two tables `inp0` (Dress Sales) and `inp1` (Attributes) and its prompts sometimes mixed them up. They are renamed `sales_*` / `attr_*` / `df` here.

## 1. Data types and variable types
**Q (Price):** what are the data type and variable type of `Price`?
**Q (Recommendation):** the same question for `Recommendation`.
**Q (unused columns):** is any column of the attribute table useless?

In [ ]:
display(attr_raw.dtypes.to_frame("dtype"))
print("Price values         :", sorted(attr_raw["Price"].dropna().unique()))
print("Recommendation values:", sorted(attr_raw["Recommendation"].unique().tolist()))
attr_raw.head()

**Answers**
- **Price:** `object` dtype, categorical **ordinal**. The labels Low < Average < Medium < High < very-high have a natural order.
- **Recommendation:** `int64` dtype holding only 0/1, so **integer type, categorical** (a binary flag, not a quantity).
- **Unused columns:** **none**. `Dress_ID` is the join key to the sales table, and every other column is used later (Price, Size, Season, NeckLine, Material, ... all appear in the grouping or bivariate steps).

## 2. Data cleaning

### 2.1 Standardise `Size`
**Q:** after expanding the abbreviations (M, L, XL, free, S/s/small), what are the lowest, highest and "Small" percentages?

*Original bug:* the course-style code `replace(['s'] and ['S'] and ['small'], 'Small')` evaluates to `replace(['small'], 'Small')` because Python's `and` returns its last operand, so `S` and `s` were never converted and Size kept 7 labels instead of 5. A lookup dictionary fixes this.

In [ ]:
SIZE_MAP = {"s": "Small", "small": "Small", "m": "Medium", "medium": "Medium",
            "l": "Large", "large": "Large", "xl": "Extra large", "extra large": "Extra large",
            "free": "Free"}

attr = attr_raw.copy()
print("Size labels before:", sorted(attr["Size"].unique()))
attr["Size"] = attr["Size"].astype(str).str.strip().str.lower().map(SIZE_MAP)
assert attr["Size"].notna().all(), "Unmapped Size label: extend SIZE_MAP"

size_pct = attr["Size"].value_counts(normalize=True).mul(100).round(2)
display(size_pct.to_frame("percent of dresses"))
print(f"Lowest : {size_pct.idxmin()} ({size_pct.min():.2f}%)")
print(f"Highest: {size_pct.idxmax()} ({size_pct.max():.2f}%)")
print(f"Small  : {size_pct['Small']:.2f}%")

**Answer** (derived from the original counts of 479 dresses: Medium 171, Free 165, Large 93, Small 36 = S 34 + s 1 + small 1, Extra large 14): **lowest = Extra large (2.92%), highest = Medium (35.70%), Small = 7.52%.** The cell above prints the same values.

### 2.2 `Dress Sales`: make the date columns numeric
Six date columns were read as `object` because they hold text tokens. Inspect them before coercing anything.

In [ ]:
DATE_COLS = [c for c in sales_raw.columns if c != "Dress_ID"]

non_numeric = {}
for c in DATE_COLS:
    coerced = pd.to_numeric(sales_raw[c], errors="coerce")
    bad = sales_raw.loc[coerced.isna() & sales_raw[c].notna(), c]
    if len(bad):
        non_numeric[c] = bad.value_counts().to_dict()
print("Non-numeric tokens found, by column:")
for c, tokens in non_numeric.items():
    print(f"  {c}: {tokens}")

sales = sales_raw.copy()
sales[DATE_COLS] = sales[DATE_COLS].apply(pd.to_numeric, errors="coerce")
assert sales["Dress_ID"].notna().all()
sales["Dress_ID"] = sales["Dress_ID"].astype("int64")
print("\nDress_ID unique -> sales:", sales["Dress_ID"].is_unique, "| attributes:", attr["Dress_ID"].is_unique)

The tokens (`removed`, `Removed`, `Orders`) are not numbers, so `to_numeric(errors="coerce")` turns them into missing values (one dress in each of six columns). It replaces five near-identical `loc[... == 'removed']` cells in the original.

### 2.3 `Dress Sales`: drop date columns with more than 40% missing values

In [ ]:
missing_pct = sales[DATE_COLS].isnull().mean().mul(100).round(1)
display(missing_pct[missing_pct > 0].to_frame("missing %"))

MISSING_DROP_THRESHOLD = 0.40
cols_dropped = [c for c in DATE_COLS if sales[c].isnull().mean() > MISSING_DROP_THRESHOLD]
sales = sales.drop(columns=cols_dropped)
DATE_COLS_KEPT = [c for c in DATE_COLS if c not in cols_dropped]
print("Dropped (>40% missing):", cols_dropped)
print("Date columns kept     :", len(DATE_COLS_KEPT))

### 2.4 Aggregate the date columns into four seasons
The course prescribes: Jun-Aug = Summer, Sep-Nov = Autumn, Dec-Feb = Winter, Mar-May = Spring, reading each column label as **day-month-2013**. The mapping is written out explicitly below so it can be audited (and changed in one place).

In [ ]:
SEASON_COLS = {
    "Summer": ["09-06-2013", "10-06-2013", "29-08-2013", "31-08-2013", "09-08-2013"],
    "Autumn": ["09-10-2013", "14-09-2013", "16-09-2013", "18-09-2013", "20-09-2013",
               "22-09-2013", "24-09-2013", "28-09-2013"],
    "Winter": ["09-02-2013", "09-12-2013", "10-12-2013"],
    "Spring": ["09-04-2013"],
}
SEASONS = ["Summer", "Autumn", "Winter", "Spring"]

used = [c for cols in SEASON_COLS.values() for c in cols]
assert sorted(used) == sorted(DATE_COLS_KEPT), "each kept date column must map to exactly one season"

for season, cols in SEASON_COLS.items():
    sales[season] = sales[cols].sum(axis=1, skipna=False)      # NaN if any date is missing, as in the original
season_totals = sales[SEASONS].sum()

# regression check against the original run (soft: prints, does not stop)
ORIGINAL_SEASON_TOTALS = {"Summer": 691907.0, "Autumn": 1353543.0, "Winter": 473640.0, "Spring": 143600.0}
check = pd.DataFrame({"this run": season_totals, "original run": pd.Series(ORIGINAL_SEASON_TOTALS)})
check["match"] = check["this run"] == check["original run"]
display(check.loc[SEASONS])

In [ ]:
# How many date columns does each "season" get? Totals are mostly a function of this.
per_col = pd.DataFrame({"date_columns": pd.Series({s: len(c) for s, c in SEASON_COLS.items()}),
                        "total_sales": season_totals})
per_col["avg_per_date_column"] = (per_col["total_sales"] / per_col["date_columns"]).round(0)
display(per_col.loc[SEASONS])

> **Caveat that affects every season-level result (read before trusting them).**
> 1. **The season totals mostly reflect how many date columns each season received** (Summer 5, Autumn 8, Winter 3, Spring 1). Per column the averages are close: about 138k (Summer), 169k (Autumn), 158k (Winter), 144k (Spring).
> 2. **The column labels look like mixed-format dates, not Feb/Apr/Jun...** The UCI description says sales were monitored on *alternate days*. Read that way, the 23 labels run every second day from 29 Aug to 12 Oct 2013 (e.g. `09-02-2013` = 2 Sep, `10-12-2013` = 12 Oct), so the whole table sits in late summer / early autumn and there is no real winter or spring data.
> 3. **The values look cumulative.** In every displayed row the counts only increase across columns in file order, and so do the column means. If so, adding columns together sums running totals and "Sales" is an index, not units sold.
>
> The check below tests points 2-3 on all rows. The course's mapping is kept because the exercise requires it; treat season comparisons as an artefact of the exercise.

In [ ]:
ordered = sales[DATE_COLS_KEPT]                         # columns in file order
step = ordered.diff(axis=1).iloc[:, 1:]
never_decreases = (step.fillna(0) >= 0).all(axis=1)
print(f"Dresses whose count never decreases across the date columns: {never_decreases.mean():.1%}")
print("Column means increase in file order:", ordered.mean().is_monotonic_increasing)

### 2.5 Merge sales onto attributes (left join, so every sales row is kept)

In [ ]:
season_df = sales[["Dress_ID"] + SEASONS]
merged = season_df.merge(attr, how="left", on="Dress_ID")
print("rows before / after merge:", len(season_df), "/", len(merged))
print("dresses with no attribute row:", int(merged["Style"].isnull().sum()))
if len(merged) != len(season_df):
    warnings.warn("The merge changed the row count: check for duplicate Dress_ID values")
nulls = merged.isnull().sum()
display(nulls[nulls > 0].to_frame("nulls"))

### 2.6 Missing values in the merged table
Two kinds, handled differently:
- **Type 1 (1-3 missing each):** `Price`, `Season`, `NeckLine`, `SleeveLength`, `Winter`, `Autumn`. Too few to matter, so those dresses are dropped.
- **Type 2 (21-53% missing):** `Material`, `FabricType`, `Decoration`, `Pattern Type`. Dropping rows would discard most of the data, so missing becomes its own category, `Missing`.

*Original bug:* `inp3.Material.replace(np.NaN, "Missing")` was never assigned back, so the label was never applied.

In [ ]:
TYPE1 = ["Price", "Season", "NeckLine", "SleeveLength", "Winter", "Autumn"]
TYPE2 = ["Material", "FabricType", "Decoration", "Pattern Type"]

display(merged[TYPE2].isnull().mean().mul(100).round(1).to_frame("missing %"))
df = merged.dropna(subset=TYPE1).copy()
dropped = len(merged) - len(df)
print(f"Type 1: dropped {dropped} of {len(merged)} dresses ({dropped / len(merged):.1%})")

### 2.7 Standardise category labels
**Q:** which columns have irregular category names? **Answer: `Season` and `SleeveLength` are the clearest cases** (Season: `Automn`, `spring`, `winter` next to `Autumn`, `Spring`, `Winter`; SleeveLength: `sleevless`, `sleeevless`, `sleveless`, `sleeveless`, `threequater`, `thressqatar`, ...). The other text columns also carry case differences and typos, so all are standardised. The original template left this step empty, and Season split a single heat-map into 7 rows.

Only unambiguous spelling fixes are applied. Judgement calls (`sill`, `model`, `chiffonfabric`) are left as they are.

In [ ]:
TEXT_COLS = ["Style", "Season", "NeckLine", "SleeveLength", "Material", "FabricType", "Decoration", "Pattern Type"]
labels_before = {c: df[c].nunique() for c in TEXT_COLS}

FIXES = {   # applied after strip + lower-case + 'collor' -> 'collar'
    "Season":       {"automn": "autumn"},
    "NeckLine":     {"bowneck": "bow-neck", "sqare-collar": "square-collar", "turndowncollar": "turndown-collar"},
    "SleeveLength": {"sleevless": "sleeveless", "sleeevless": "sleeveless", "sleveless": "sleeveless",
                     "threequater": "threequarter", "thressqatar": "threequarter",
                     "capsleeves": "cap-sleeves", "half": "halfsleeve"},
    "Material":     {"polyster": "polyester", "shiffon": "chiffon", "viscos": "viscose"},
    "FabricType":   {"shiffon": "chiffon", "sattin": "satin", "flannael": "flannel",
                     "wollen": "woolen", "knitting": "knitted"},
    "Decoration":   {"embroidary": "embroidery"},
    "Pattern Type": {"leapord": "leopard"},
}
for col in TEXT_COLS:
    s = df[col].str.strip().str.lower().str.replace("collor", "collar", regex=False)
    df[col] = s.replace(FIXES.get(col, {}))
df["Season"] = df["Season"].str.capitalize()

labels_after = {c: df[c].nunique() for c in TEXT_COLS}
display(pd.DataFrame({"labels before": labels_before, "labels after": labels_after}))
assert set(df["Season"].unique()) == set(SEASONS), df["Season"].unique()

df[TYPE2] = df[TYPE2].fillna("Missing")          # Type-2 missing values get their own label
print("Nulls left in the analysis table:", int(df.isnull().sum().sum()))

*Note:* `SleeveLength` contains `turndowncollar` / `urndowncollar`, which are **neckline** values typed into the wrong column. They are tiny and fall into "Others" below.

### 2.8 Group rare categories into "Others"
Rule from the course: a category whose **total sales across all seasons** is below a threshold becomes "Others" (50,000 for Style, NeckLine, SleeveLength; 25,000 for Material, FabricType, Pattern Type, Decoration). `Missing` is never grouped.

*Original bug:* the course-style code used `inp3.replace(list_of_labels, 'Others', inplace=True)` on the **whole DataFrame**, so a label in one column also overwrote identical labels in other columns. Evidence in the original run: `FabricType` already contained an `Others` category (30,819 sales) *before* its own grouping step, because `shiffon` and `other` were in the Material list. `lace` (a Material label, and also the most common Decoration at 26%) was exposed to the same problem. The function below changes one column only.

In [ ]:
df["Sales"] = df[SEASONS].sum(axis=1)            # total sales of a dress across the four "seasons"

def group_rare(frame, col, min_sales, label="Others", keep=("Missing",)):
    """Relabel categories of ONE column whose total sales are below min_sales."""
    by_cat = frame.groupby(col)["Sales"].sum()
    rare = [c for c in by_cat[by_cat < min_sales].index if c not in keep]
    frame[col] = frame[col].where(~frame[col].isin(rare), label)
    return sorted(rare)

RARE_RULES = [("Style", 50000), ("NeckLine", 50000), ("SleeveLength", 50000),
              ("Material", 25000), ("FabricType", 25000), ("Pattern Type", 25000), ("Decoration", 25000)]
rare_log = {}
for col, threshold in RARE_RULES:
    rare_log[col] = group_rare(df, col, threshold)
for col, rare in rare_log.items():
    print(f"{col:13s} -> Others: {rare}")

# regression check: total sales after the 8 dropped dresses, original run = 2,635,869
print("\nDresses:", len(df), "| total sales:", f"{df['Sales'].sum():,.0f}", "(original run: 471 dresses, 2,635,869)")

**Q (Style):** which Style categories fall below 50,000 sales? **Answer: Flare, Novelty, OL and fashion** (2,089 / 13,286 / 289 / 395 sales, 16,059 together). Of the course's options, *Novelty, fashion, Flare* is the one that lists only qualifying categories. Bohemian (69,117) and work (93,095) stay above the threshold.

## 3. Univariate analysis

### 3.1 Categorical, unordered: `Style`
**Q:** which variable is unordered? **Answer: `Style`** (a nominal label with no ranking). Price and Size have a natural order.

In [ ]:
style_pct = df["Style"].value_counts(normalize=True).mul(100).round(1)
display(style_pct.to_frame("% of dresses"))
print(f"cute = {style_pct.get('cute', float('nan')):.1f}% of dresses   |   Others = {style_pct.get('Others', float('nan')):.1f}% of dresses")

style_sales_pct = df.groupby("Style")["Sales"].sum().pipe(lambda s: s.div(s.sum()).mul(100))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
style_pct.sort_values().plot.barh(ax=ax[0], title="Share of dresses by style (%)")
style_sales_pct.sort_values().plot.barh(ax=ax[1], title="Share of sales by style (%)")
plt.tight_layout()
plt.show()

*The original run never printed the Style percentages: the cell labelled "percentage of each category in Style" actually contained NeckLine code. The cell above answers the course question (the percentages of `cute` and `Others`).*

### 3.2 Categorical, ordered: `Price` and `Size`

In [ ]:
PRICE_ORDER = ["Low", "Average", "Medium", "High", "very-high"]
SIZE_ORDER = ["Small", "Medium", "Large", "Extra large", "Free"]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df["Price"].value_counts().reindex(PRICE_ORDER).plot.bar(ax=ax[0], title="Dresses by price band")
df["Size"].value_counts().reindex(SIZE_ORDER).plot.bar(ax=ax[1], title="Dresses by size")
plt.tight_layout()
plt.show()

*Original run:* Average 50.3%, Low 34.4%, Medium 6.4%, High 4.5%, very-high 4.5% of dresses are in each price band. Half the catalogue is one band.

### 3.3 Numerical: seasonal sales
**Q:** (a) the gap between the maximum and the 75th percentile of Autumn; (b) which season has the largest gap between its maximum and its 99th percentile?

In [ ]:
display(df[SEASONS].describe().round(0))

fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
for a, season in zip(ax, SEASONS):
    df[season].plot.box(ax=a, title=season)
plt.tight_layout()
plt.show()

tail = pd.DataFrame({"p75": df[SEASONS].quantile(0.75), "p99": df[SEASONS].quantile(0.99), "max": df[SEASONS].max()})
tail["max - p75"] = tail["max"] - tail["p75"]
tail["max - p99"] = tail["max"] - tail["p99"]
display(tail.round(0))
print("Largest (max - p99) gap:", tail["max - p99"].idxmax())

**Answers** (from the original run, which these cells reproduce)
- **(a) Autumn:** max 55,532 - 75th percentile 3,126 = **52,406, about 52,000**.
- **(b) Autumn** has the largest max-to-p99 gap (31,398), ahead of Summer (21,603), Winter (11,403) and Spring (4,717).
- **Shape:** every season is heavily right-skewed. The mean is 2.1-2.8x the median (Autumn mean 2,853 vs median 1,334), so a few blockbusters dominate.

In [ ]:
# How concentrated are sales?
top10_share = df["Sales"].nlargest(int(round(0.10 * len(df)))).sum() / df["Sales"].sum()
best = df.loc[df["Sales"].idxmax()]
print(f"Top 10% of dresses = {top10_share:.1%} of all sales")
print(f"Best-selling dress: {best['Sales']:,.0f} sales = {best['Sales'] / df['Sales'].sum():.1%} of all sales "
      f"(style: {best['Style']}, price: {best['Price']}, neckline: {best['NeckLine']})")
print(f"Rating == 0 for {(df['Rating'] == 0).mean():.1%} of dresses")

*Original run:* the four seasonal maxima (55,532 + 35,577 + 20,388 + 7,374) add up to **118,871, exactly the total of the one "ruffled"-neckline dress (0.2% of dresses)**. A single dress is therefore **about 4.5% of all sales**.

## 4. Bivariate analysis

### 4.1 Numerical vs categorical
**Q:** which `Price` band has the lowest average `Rating`? **Q:** what is the median `Rating` of `vintage` dresses? **Q:** which season has the highest mean sales among recommended dresses (`Recommendation == 1`)?

In [ ]:
rating_by_price = df.groupby("Price")["Rating"].agg(["mean", "median", "count"]).reindex(PRICE_ORDER).round(2)
display(rating_by_price)
print("Lowest mean rating:", rating_by_price["mean"].idxmin())

# sensitivity: Rating == 0 probably means "not rated", so repeat without the zeros
rated_only = df[df["Rating"] > 0].groupby("Price")["Rating"].mean().reindex(PRICE_ORDER).round(2)
display(rated_only.to_frame("mean rating, Rating > 0 only"))

**Answer:** **High** has the lowest mean rating (**2.91**), ahead of very-high 3.12, Medium 3.16, Average 3.45 and Low 3.73 (original run). Caution: ratings of exactly 0.0 appear on dresses with large sales (e.g. dress 1212192089, 30,682 sales, rating 0.0), which suggests 0 means "not rated" and drags every mean down; the second table repeats the comparison without them.

In [ ]:
median_by_style = df.groupby("Style")["Rating"].median()
display(median_by_style.to_frame("median rating"))
print("vintage median rating:", median_by_style.get("vintage", float("nan")))

*The original run did not display this value (the median table was not the last expression in its cell, so only `head()` printed). The cell above prints it.*

In [ ]:
rec_sales = df.groupby("Recommendation")[SEASONS].mean().round(1)
display(rec_sales)
print("Season with the highest mean sales among recommended dresses:", rec_sales.loc[1].idxmax())
print(f"Share of dresses recommended: {df['Recommendation'].mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.boxplot(data=df, x="Recommendation", y="Autumn", ax=ax)
ax.set_title("Autumn sales by recommendation")
plt.tight_layout()
plt.show()

**Answer:** the original run showed recommended-dress means of Summer 1,470, Winter 1,003 and Spring 304 (the Autumn mean was not printed because the cell ended in a plot). Autumn's *overall* mean (2,853) is already above all three, so **Autumn** is the expected answer; the cell above prints the exact value.

**Observation:** recommended and non-recommended dresses sell almost the same. Means differ by only +2.5% (Summer), +1.5% (Winter) and +1.8% (Spring), tiny against standard deviations in the thousands.

### 4.2 Categorical vs categorical: `Size` vs `Recommendation`
**Q:** which sizes have the highest recommendation rate?

In [ ]:
rec_by_size = df.groupby("Size")["Recommendation"].agg(["mean", "count"]).reindex(SIZE_ORDER).round(3)
display(rec_by_size)

fig, ax = plt.subplots(figsize=(6, 3.5))
rec_by_size["mean"].plot.barh(ax=ax, title="Share of dresses recommended, by size")
plt.tight_layout()
plt.show()

**Answer: Free and Medium** (about 0.46 and 0.45 in the original bar chart, read off the plot), then Small (~0.38) and Large (~0.36); **Extra large is lowest (~0.29)**. Extra large has only 14 dresses, so its rate is noisy. In the original chart `s` and `small` showed as empty bars (one dress each, none recommended), a symptom of the unfixed Size labels.

## 5. Multivariate analysis

### 5.1 `Style` x `Price` x `Recommendation`
**Q:** which Style / Price pair has the highest share of recommended dresses? Cell counts are shown beside the rates, because many cells are tiny.

In [ ]:
style_price = pd.pivot_table(df, index="Price", columns="Style", values="Recommendation").reindex(PRICE_ORDER)
n_style_price = pd.pivot_table(df, index="Price", columns="Style", values="Recommendation", aggfunc="count").reindex(PRICE_ORDER)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sns.heatmap(style_price, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax[0])
ax[0].set_title("Share of dresses recommended")
sns.heatmap(n_style_price, annot=True, fmt=".0f", cmap="Blues", ax=ax[1])
ax[1].set_title("Number of dresses in the cell (n)")
plt.tight_layout()
plt.show()

pairs = (df.groupby(["Price", "Style"])["Recommendation"]
           .agg(share_recommended="mean", n="count")
           .reset_index()
           .sort_values(["share_recommended", "n"], ascending=False))
display(pairs.head(10))

**Answer: Price *Medium* and Style *cute*** (a 100% recommendation rate in the original run). It beats the other offered pairs: very-high party 0.82, low sexy 0.75, medium vintage 0.50. Treat it as **exploratory**: cells with a handful of dresses jump to 0% or 100%, and `bohemian` ties at 100% in two price bands. Check the count heat-map.

### 5.2 `Season` x `Material` x `Recommendation`
**Q:** which materials have no recommendation in Summer and Winter?

In [ ]:
season_material = pd.pivot_table(df, index="Season", columns="Material", values="Recommendation").reindex(SEASONS)
fig, ax = plt.subplots(figsize=(11, 3.5))
sns.heatmap(season_material, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
ax.set_title("Share of dresses recommended, Season x Material")
plt.tight_layout()
plt.show()

no_rec = [m for m in season_material.columns
          if all(pd.isna(season_material.loc[s, m]) or season_material.loc[s, m] == 0 for s in ("Summer", "Winter"))]
print("Materials with no recommendation in Summer and Winter:", no_rec)
display(season_material.loc[["Summer", "Winter"], no_rec])

**Answer: Milksilk and Microfiber.** In the original pivot, microfiber was 0.0 in both Summer and Winter, and milksilk was 0.0 in Summer and Winter (its Winter value hid in the lower-case `winter` row until labels were standardised). The Season labels in that original pivot were split across 7 rows (`Automn`, `Autumn`, `spring`, `Spring`, `winter`, `Winter`, `Summer`), which is why this step needs the standardisation in 2.7.

## 6. Findings (supported by the outputs)
Figures come from the original run's outputs, or from exact arithmetic on them; the cells above reprint them. **Re-run the notebook top to bottom and confirm they match before publishing.**

| # | Finding | Evidence | Confidence |
|---|---|---|---|
| 1 | **Most of the effort is data quality.** Seven spellings for four seasons; seven Size labels for five sizes; `Sexy`/`sexy`; text tokens in six sales columns; 6 of 23 sales columns more than 40% missing (46-54%); FabricType 53%, Decoration 47%, Material 25%, Pattern Type 21% missing. | 2.1-2.7 | High |
| 2 | **Sales are extremely concentrated and right-skewed.** Mean is 2.1-2.8x the median in every season; one "ruffled"-neckline dress is about 4.5% of all sales (118,871 of 2,635,869). | 3.3 | High |
| 3 | **Casual and o-neck dominate.** Casual is about 42% of sales, Sexy (both spellings) about 20%, cute about 14%; o-neck is 54% of dresses and about 63% of sales, v-neck about 21%. | 2.8, 3.1 | High (descriptive) |
| 4 | **Cheap dresses are rated best, expensive ones lower.** Mean rating: Low 3.73, Average 3.45, Medium 3.16, very-high 3.12, High 2.91. Price mix is 50% Average, 34% Low, 15% the rest. | 4.1 | Medium (0-ratings may mean "not rated") |
| 5 | **Recommendation is almost unrelated to sales volume.** Recommended dresses sell 1.5-2.5% more on average in the three seasons printed in the original run, tiny next to the spread. | 4.1 | Medium |
| 6 | **Free and Medium sizes are recommended most (about 45%); Extra large least (about 29%).** | 4.2 | Low-medium (read from a chart; XL has 14 dresses) |
| 7 | **Medium-price "cute" dresses are always recommended; very-high "party" 82%.** | 5.1 | Low (exploratory, small cells) |

## 7. Limitations
1. **Season totals are an artefact (most important).** Under the course's day-first reading Summer gets 5 date columns, Autumn 8, Winter 3 and Spring 1, so Autumn "wins" (50.8% of all season totals) mainly because it has the most columns. Per column the totals are close (about 138k-169k). The UCI note that sales were monitored on alternate days, and the increasing order of the columns, point to dates running 29 Aug-12 Oct 2013, so there is no genuine winter or spring. **No conclusion about seasonality should be drawn.**
2. **"Sales" may be a sum of cumulative counts.** If the date columns are running totals, adding them double-counts and gives an index of popularity, not units sold. The check in 2.4 tests this on every row.
3. **Short, single-retailer snapshot.** About six weeks of 2013 for one catalogue. Nothing here generalises to other retailers or years.
4. **`Recommendation` is undefined.** The source says only that recommendations are "according to their sales". How it was assigned is not documented, so its relationship to sales (finding 5) may be partly by construction.
5. **`Rating` of 0.0 is ambiguous** (likely "not rated"). Mean ratings by price or style include these zeros unless filtered; 4.1 shows both versions.
6. **Missing-data choices are judgement calls.** Eight dresses (1.7%) were dropped for 1-3 missing values; Type-2 gaps (21-53%) became a `Missing` label, and any dress with a `removed`/`Orders` token has NaN for that season and is dropped. Different choices could move the numbers.
7. **Grouping thresholds are arbitrary.** The 50,000 / 25,000 sales cut-offs come from the course and apply to *sales*, not counts. Raising or lowering them changes the "Others" bucket.
8. **Small cells and no inference.** Cross-tabs have many cells with a handful of dresses; there are no confidence intervals or significance tests. Everything is descriptive association, not cause.
9. **Data differ from the original release.** The course CSVs hold 479 dresses; UCI lists 500/501. It is unknown which rows were removed.
10. **Ambiguous labels were left alone** (`sill`, `model`, `chiffonfabric`, `none` vs `plain`), and misplaced values (neckline terms in `SleeveLength`) fall into "Others".

In [ ]:
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)